# Lecture 3: Linear Programming

---

```{note}
This lecture opens the Linear Programming module. We move from classifying optimization problems (Lecture 2) to actually building them: translating a messy real-world transportation scenario into a precise, solvable LP model. The graphical method, simplex algorithm, sensitivity analysis, and duality follow in subsequent lectures.
```

**Learning Objectives**

By the end of this lecture, you will be able to:
1. State the four structural assumptions of LP and verify whether a given problem satisfies them.
2. Formulate the objective function, decision variables, and constraints for a transportation problem as a linear program in standard form.
3. Identify the feasible region of a two-variable LP and locate its corner-point solutions graphically.

**Prerequisites**: Lecture 1 — Five-Step Process; Lecture 2 — Problem Types.  

**Estimated time**: 50 minutes.

---

## Intuition

Every logistics manager, fleet operator, and transport planner faces the same fundamental tension: limited resources, competing demands, and a measurable goal. How many trucks do you rent? How do you split traffic across two routes? How many buses of each type maximize your profit?

When the relationship between decisions and outcomes is *proportional* and *additive* — that is, doubling the number of trucks doubles the cost, and the cost of trucks and drivers adds up independently — the problem becomes a Linear program (LP). This is not a simplification made for convenience; it is often an accurate description of the real world at the operational scale where unit costs are fixed, capacities are linear, and resources don't interact.

> **Why linear programming matters**: LP is the workhorse of applied optimization. It is computationally tractable even at enormous scale (millions of variables), has a rich theory (duality, sensitivity analysis) that yields managerial insight, and underpins more complex methods like branch-and-bound for integer programs and interior-point methods for NLPs. Mastering LP formulation is the single most important skill in this module.

Consider the Chennai Metropolitan Transport Corporation (MTC) planning its morning peak bus deployment across two corridors. MTC knows the cost per bus trip, the passenger capacity of each bus type, and the minimum service frequency each corridor requires. The question — how many buses of each type to deploy — has a linear cost function and linear capacity constraints. That is an LP.

## Formal Definition and Standard Form

**Definition**: A Linear Program (LP) is an optimization problem in which (i) the objective function is a linear combination of the decision variables, and (ii) every constraint is a linear equality or inequality in the decision variables.

### Standard Form

A general LP with $n$ decision variables and $m$ constraints is written in standard form as:

$$
\max_{\mathbf{x}} \quad f(\mathbf{x}) = c_1 x_1 + c_2 x_2 + \cdots + c_n x_n
$$

Subject to:

$$
\begin{aligned}
a_{11}x_1 + a_{12}x_2 + \cdots + a_{1n}x_n &\leq b_1 \\
a_{21}x_1 + a_{22}x_2 + \cdots + a_{2n}x_n &\leq b_2 \\
&\vdots \\
a_{m1}x_1 + a_{m2}x_2 + \cdots + a_{mn}x_n &\leq b_m \\
x_i &\geq 0 \quad \forall\, i \in \{1, \ldots, n\}
\end{aligned}
$$

In compact matrix–vector notation, with $\mathbf{c} \in \mathbb{R}^n$, $\mathbf{A} \in \mathbb{R}^{m \times n}$, $\mathbf{b} \in \mathbb{R}^m$:

$$
\max_{\mathbf{x} \geq \mathbf{0}} \quad \mathbf{c}^\top \mathbf{x} \qquad \text{subject to} \qquad \mathbf{A}\mathbf{x} \leq \mathbf{b}
$$

### Notation

| Symbol | Meaning |
|--------|---------|
| $x_i$ | Decision variable $i$ |
| $c_i$ | Objective function coefficient (cost/profit per unit of $x_i$) |
| $a_{ij}$ | Coefficient of $x_j$ in constraint $i$ |
| $b_i$ | Right-hand side (RHS) of constraint $i$ |
| $\mathbf{A}$ | Constraint matrix ($m \times n$) |
| $f(\mathbf{x})$ | Objective function value |

### Constraint Conversions to Standard Form

Any LP can be rewritten in this form using the following conversions:

| Original | Conversion to standard form |
|----------|-----------------------------|
| $\min\ f(\mathbf{x})$ | $\max\ {-f(\mathbf{x})}$ (negate objective; negate result at the end) |
| $g(\mathbf{x}) \geq b$ | $-g(\mathbf{x}) \leq -b$ (negate both sides) |
| $g(\mathbf{x}) = b$ | $g(\mathbf{x}) \leq b$ **and** $-g(\mathbf{x}) \leq -b$ |
| $x_i \leq 0$ | substitute $x_i' = -x_i \geq 0$ |
| $x_i$ unrestricted | substitute $x_i = x_i^+ - x_i^-$, both $\geq 0$ |

### Assumptions

An LP is valid — and its solutions meaningful — only when the following four assumptions hold:

| Assumption | Statement | Violated when |
|------------|-----------|---------------|
| **Proportionality** | The contribution of $x_i$ to the objective and constraints is directly proportional to its value. | Unit costs change with quantity (e.g., bulk discounts, congestion effects). |
| **Additivity** | The total objective value is the sum of individual contributions; no interaction terms between variables. | Joint costs exist (e.g., two vehicles on the same route share fuel). |
| **Divisibility** | Decision variables may take any non-negative real value (fractional solutions are allowed). | Only whole units make sense (trucks, buses, coaches) — ILP required. |
| **Certainty** | All parameters ($c_i$, $a_{ij}$, $b_i$) are known exactly and do not change. | Demand, costs, or capacities are uncertain — stochastic programming required. |

```{important}
Identifying which assumptions hold (and which are approximations) is not a formality — it is a risk assessment. A proportionality violation (e.g., treating a quadratic cost as linear) can produce a recommendation that is optimal for the model but infeasible or costly in practice. Always document your assumptions.
```

### Subtypes: ILP and MILP

When physical meaning requires the decision variables to take integer values (e.g., number of trucks, number of buses), the problem becomes:

- Integer Linear Program (ILP): all $x_i \in \mathbb{Z}_+$ — requires branch-and-bound or cutting-plane methods.
- Mixed-Integer Linear Program (MILP): some $x_i \in \mathbb{Z}_+$, some $x_i \in \mathbb{R}_+$ — common in network design problems.

The LP relaxation of an ILP (dropping the integrality requirement) provides both a lower bound and a starting point for integer solution methods. We treat all examples here as LP relaxations unless otherwise stated.

---

## In-Class Exercises 

The three examples below progress in complexity: from a single-constraint freight allocation, to a multi-constraint traffic routing problem, to a revenue-maximization bus deployment problem with amortised fixed costs. Each follows the same formulation discipline: **define decisions → write objective → write constraints → confirm standard form**.

### Exercise 1

A textile firm based in Kochi must ship 100 tons of goods from Kanchipuram. Two truck types are available for rental:

| Truck type | Capacity (tons) | Cost per trip (₹) |
|------------|-----------------|--------------------|
| $\text{T}_1$ | 10 | 5,000 |
| $\text{T}_2$ | 20 | 8,000 |

Managerial constraints: total fleet $\leq 20$ trucks; no more than 12 trucks of any single type.

**Step 1 — Decision variables**: Let $x_1, x_2 \in \mathbb{Z}_+$ denote the number of $\text{T}_1$ and $\text{T}_2$ trucks deployed.

**Step 2 — Objective** (minimize total cost):

$$
\min_{x_1,\, x_2} \quad z = 5000x_1 + 8000x_2
$$

**Step 3 — Constraints**:

$$
\begin{aligned}
10x_1 + 20x_2 &\geq 100 & &\text{(minimum payload)}\\
x_1 + x_2 &\leq 20 & &\text{(fleet size limit)}\\
x_1 &\leq 12 & &\text{(T}_1\text{ limit)}\\
x_2 &\leq 12 & &\text{(T}_2\text{ limit)}\\
x_1,\, x_2 &\in \mathbb{Z}_+
\end{aligned}
$$

```{note}
Constraint (2) is $\leq$; in standard form it becomes $-x_1 - x_2 \geq -20$. Constraints (3) and (4) convert similarly. This is handled automatically by solvers — the conversion is shown here for pedagogical completeness.
```

---

### Exercise 2

Chennai Metropolitan Transport Corporation (MTC) operates two bus services on the Chennai Central–Tambaram corridor during morning peak hours (6 AM–10 AM):

| Service | Passengers per trip | Operating cost per trip (₹) | Trips per hour |
|---------|--------------------|-----------------------------|----------------|
| $\text{B}_1$ — Ordinary | 60 | 1,200 | — |
| $\text{B}_2$ — Air-conditioned | 40 | 2,500 | — |

MTC must schedule at least 500 ordinary passenger trips and 300 AC passenger trips per hour to meet minimum service standards. Operational constraints limit total bus trips to 15 per hour across both services, and no single service may exceed 10 trips per hour.

**Step 1 — Decision variables**: Let $x_1, x_2 \geq 0$ denote the number of trips per hour for $\text{B}_1$ and $\text{B}_2$ respectively.

**Step 2 — Objective** (minimize total hourly operating cost):

$$
\min_{x_1,\, x_2} \quad z = 1200x_1 + 2500x_2
$$

**Step 3 — Constraints**:

$$
\begin{aligned}
60x_1 &\geq 500 & &\text{(minimum ordinary passenger trips)}\\
40x_2 &\geq 300 & &\text{(minimum AC passenger trips)}\\
x_1 + x_2 &\leq 15 & &\text{(total trip limit)}\\
x_1 &\leq 10 & &\text{(B}_1\text{ trip limit)}\\
x_2 &\leq 10 & &\text{(B}_2\text{ trip limit)}\\
x_1,\, x_2 &\geq 0
\end{aligned}
$$

### Exercise 3

The Bengaluru Smart City Traffic Management Centre must allocate 3,000 vehicles per hour during the evening peak between Silk Board Junction and Hebbal across three parallel corridors:

| Corridor | Travel time (min) | Capacity (veh/hr) |
|----------|-------------------|-------------------|
| $\text{C}_1$ — Outer Ring Road (ORR) | 35 | 1,800 |
| $\text{C}_2$ — NH-44 (National Highway) | 50 | 1,200 |
| $\text{C}_3$ — Intermediate Ring Road (IRR) | 45 | 900 |

To prevent any single corridor from bearing the full load, traffic authorities require at least 400 vehicles per hour on each corridor. The objective is to minimize total vehicle hours travelled (VHT).

**Step 1 — Decision variables**: Let $x_1, x_2, x_3 \geq 0$ denote vehicles per hour assigned to $\text{C}_1$, $\text{C}_2$, and $\text{C}_3$ respectively.

**Step 2 — Objective** (minimize VHT, in vehicle-minutes per hour):

$$
\min_{x_1,\, x_2,\, x_3} \quad z = 35x_1 + 50x_2 + 45x_3
$$

**Step 3 — Constraints**:

$$
\begin{aligned}
x_1 + x_2 + x_3 &= 3000 & &\text{(serve all demand)}\\
x_1 &\leq 1800 & &\text{(C}_1\text{ capacity)}\\
x_2 &\leq 1200 & &\text{(C}_2\text{ capacity)}\\
x_3 &\leq 900 & &\text{(C}_3\text{ capacity)}\\
x_1,\, x_2,\, x_3 &\geq 400 & &\text{(minimum corridor flow)}
\end{aligned}
$$

```{note}
This is a **three-variable LP** — the feasible region lives in $\mathbb{R}^3$ and cannot be plotted on a 2D graph directly. We solve it with `linprog` and inspect the corner-point solution numerically. This is the typical situation in practice: most real LPs have far more than two variables, and solvers — not graphs — are the primary tool.
```

---

## Take-Away Exercise

Work through the following:
1. Define the decision variables.
2. Write the objective function.
3. Write all constraints in standard form.
4. Identify the type of problem (LP / ILP / MILP) and justify your choice.

### Exercise 1

BMTC operates two depot types along the Outer Ring Road (ORR) corridor:
- Depot A (Silk Board): holds up to 60 buses; costs ₹12,000/bus/day to operate.
- Depot B (Marathahalli): holds up to 80 buses; costs ₹9,000/bus/day to operate.

The minimum fleet requirement for the ORR corridor is 100 buses. Due to maintenance scheduling, Depot A must house at least 20 buses. The total operational budget is ₹12,00,000/day.

BMTC wishes to minimize total daily operating cost.

---

## Further Reading

- Hillier, F.S. and Lieberman, G.J. (2015) — *Introduction to Operations Research*, 10th ed. — McGraw-Hill *(Chapters 3–4: Introduction to LP; Solving LP)*
- Taha, H.A. (2017) — *Operations Research: An Introduction*, 10th ed. — Pearson *(Chapter 2: Modelling with LP)*
- Bazaraa, M.S., Jarvis, J.J., and Sherali, H.D. (2009) — *Linear Programming and Network Flows*, 4th ed. — Wiley *(Chapter 2: LP Formulations)*